<a href="https://colab.research.google.com/github/pavishanth-sujeevan/E.motion-/blob/llm/sarva_fine_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  CELL 1: Install & Check GPU

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets
!pip install -q -U evaluate rouge_score scikit-learn seaborn matplotlib

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

print("="*70)
print("GPU CHECK")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 124.0 MB/s eta 0:00:00
GPU CHECK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


##  CELL 2: Mount Drive & Load Tamil Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Data paths
DATA_DIR = "/content/drive/MyDrive/processed_tamil"  # ← CHANGE THIS
TRAIN_FILE = f"{DATA_DIR}/train_ta.jsonl"
VAL_FILE = f"{DATA_DIR}/val_ta.jsonl"

import os
print("\nChecking files...")
print(f"Train: {'Yes' if os.path.exists(TRAIN_FILE) else 'No'}")
print(f"Val: {'Yes' if os.path.exists(VAL_FILE) else 'No'}")

Mounted at /content/drive

Checking files...
Train: Yes
Val: Yes


##  CELL 3: Model Config - LLaMA 3.2-3B (Better than Gemma for Tamil!)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json

# FIXED MODEL CHOICE
MODEL_ID = "sarvamai/sarvam-2b-v0.5"
OUTPUT_DIR = "./sarvam-tamil-therapy-fixed"

print("="*70)
print("MODEL: sarvam-2b-v0.5")
print("="*70)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("\n Config ready")

MODEL: sarvam-2b-v0.5

 Config ready


##  CELL 4: Load LLaMA with Optimized LoRA

In [ ]:
print("="*70)
print("LOADING sarvam-2b-v0.5")
print("="*70)

print("\n1/3: Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("    Model loaded")

print("\n2/3: Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("    Tokenizer loaded")

# Test Tamil tokenization
test_tamil = "எனக்கு மிகவும் வருத்தமாக உள்ளது"
tokens = tokenizer.encode(test_tamil)
print(f"\n   Tamil test: {test_tamil}")
print(f"   Tokens: {len(tokens)}")
print(f"    Tamil works!")

print("\n3/3: Applying LoRA (optimized for Tamil)...")
peft_config = LoraConfig(
    r=96,
    lora_alpha=192,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("\n MODEL READY")

LOADING sarvam-2b-v0.5

1/3: Loading model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

    Model loaded

2/3: Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

    Tokenizer loaded

   Tamil test: எனக்கு மிகவும் வருத்தமாக உள்ளது
   Tokens: 6
    Tamil works!

3/3: Applying LoRA (optimized for Tamil)...
trainable params: 143,818,752 || all params: 2,652,653,568 || trainable%: 5.4217

 MODEL READY


##  CELL 5: Load Data with ROBUST Preprocessing

In [ ]:
print("="*70)
print("LOADING TAMIL DATA (ROBUST)")
print("="*70)

def extract_user_input_robust(prompt):
    """
    ROBUSTLY extract user input from various prompt formats.
    This prevents the 'user' output bug!
    """
    # Try pattern 1: "User: {text} Assistant:"
    if 'User:' in prompt and 'Assistant:' in prompt:
        parts = prompt.split('User:')
        if len(parts) > 1:
            user_part = parts[1]
            if 'Assistant:' in user_part:
                text = user_part.split('Assistant:')[0].strip()
                if text:  # Make sure it's not empty
                    return text

    # Try pattern 2: Just "User: {text}"
    if 'User:' in prompt:
        parts = prompt.split('User:')
        if len(parts) > 1:
            text = parts[1].strip()
            if text:
                return text

    # Fallback: return entire prompt
    return prompt.strip()

def load_tamil_data(filepath):
    """Load Tamil data with emotion extraction"""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            try:
                entry = json.loads(line)

                # Extract emotion
                prompt = entry.get('prompt', '')
                if 'feeling' in prompt.lower():
                    try:
                        emotion = prompt.lower().split('feeling')[1].split('.')[0].strip()
                        entry['emotion'] = emotion
                    except:
                        entry['emotion'] = 'unknown'
                else:
                    entry['emotion'] = 'unknown'

                data.append(entry)
            except:
                print(f"    Skipped line {i}")
    return data

# Load
print("\nLoading training data...")
train_data = load_tamil_data(TRAIN_FILE)
print(f"    {len(train_data)} samples")

print("\nLoading validation data...")
val_data = load_tamil_data(VAL_FILE)
print(f"    {len(val_data)} samples")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

# Show emotions
print("\n Emotions:")
emotions = {}
for item in train_data:
    e = item.get('emotion', 'unknown')
    emotions[e] = emotions.get(e, 0) + 1
for e, c in sorted(emotions.items(), key=lambda x: x[1], reverse=True):
    print(f"   {e}: {c}")

# FIXED FORMATTING FOR LLAMA
def format_for_llama_tamil(example):
    """
    Format Tamil data for LLaMA (NOT Gemma!).
    This FIXES the 'user' output bug.
    """
    # Extract user input ROBUSTLY
    prompt = example.get('prompt', '')
    user_query = extract_user_input_robust(prompt)

    # Get response and emotion
    bot_response = example.get('target', '')
    emotion = example.get('emotion', 'unknown')

    # Add emotion conditioning
    if emotion != 'unknown':
        user_query = f"[உணர்வு: {emotion}] {user_query}"

    # LLaMA 3.2 format (SIMPLE AND WORKS!)
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_query}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{bot_response}<|eot_id|>"
    )

    return {
        "text": text,
        "emotion": emotion,
        "user_input": user_query,
        "target_response": bot_response
    }

# Format
print("\nFormatting for LLaMA...")
train_dataset = train_dataset.map(format_for_llama_tamil)
val_dataset = val_dataset.map(format_for_llama_tamil)

# Save for eval
train_dataset_eval = train_dataset
val_dataset_eval = val_dataset

# Keep only text
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c != "text"]
)
val_dataset = val_dataset.remove_columns(
    [c for c in val_dataset.column_names if c != "text"]
)

print("\n DATA READY")
print(f"   Train: {len(train_dataset)}")
print(f"   Val: {len(val_dataset)}")

# Show sample
print("\n Sample:")
print("="*70)
print(train_dataset_eval[0]['text'][:300] + "...")
print("="*70)

LOADING TAMIL DATA (ROBUST)

Loading training data...
    788 samples

Loading validation data...
    98 samples

 Emotions:
   neutral: 114
   angry: 114
   surprised: 114
   sad: 114
   happy: 114
   fearful: 113
   disgust: 105

Formatting for LLaMA...


Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]


 DATA READY
   Train: 788
   Val: 98

 Sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|><|start_header_id|>user<|end_header_id|>

[உணர்வு: neutral] நான் ஒரு கஃபேயில் அமர்ந்திருக்கிறேன்.<|eot_id|><|start_header_id|>assistant<|...


##  CELL 6: Training Config (8 Epochs for Better Tamil)

In [ ]:
print("="*70)
print("TRAINING CONFIG")
print("="*70)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # 1. Expand Context Window
    dataset_text_field="text",
    max_length=2048,               # Increased from 1024; A100 can easily handle larger context
    packing=True,                  # Set to True for A100; higher efficiency for large datasets

    # 2. Aggressive Batching
    # A100 allows much higher per-device batches, leading to more stable gradients
    num_train_epochs=10,           # Further increased for deep script mastery
    per_device_train_batch_size=8, # Increased from 2; A100 VRAM allows this
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, # Total effective batch remains large but faster

    # 3. Learning & Regularization
    learning_rate=1e-4,            # Slightly lower for more stable convergence on long runs
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # 4. A100 Specific Optimizations
    optim="adamw_torch_fused",     # Faster than paged_adamw on A100
    weight_decay=0.05,             # Higher weight decay to prevent overfitting on 10 epochs
    max_grad_norm=1.0,

    # 5. Native High Precision
    bf16=True,                     # A100 is designed for BF16
    fp16=False,
    tf32=True,                     # Enable TensorFloat-32 for extra speed on A100

    # 6. Faster Training
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # 7. Evaluation & Save Logic
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Logging
    logging_steps=5,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TRAINING CONFIG


##  CELL 7: Start Training!

In [ ]:
print("="*70)
print("STARTING TAMIL THERAPY TRAINING")
print("="*70)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("\n Training...")

trainer.train()

print("\n TRAINING DONE!")

STARTING TAMIL THERAPY TRAINING


Adding EOS to train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/788 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

...

Packing eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



 Training...


Step,Training Loss,Validation Loss



 TRAINING DONE!


##  CELL 8: Save Model

In [ ]:
SAVE_DIR = f"{OUTPUT_DIR}/final_model"
print(f"Saving to: {SAVE_DIR}")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(" Saved!")

try:
    DRIVE_DIR = "/content/drive/MyDrive/sarva-tamil-therapy"
    model.save_pretrained(DRIVE_DIR)
    tokenizer.save_pretrained(DRIVE_DIR)
    print(f" Also saved to Drive: {DRIVE_DIR}")
except:
    print(" Drive save skipped")

Saving to: ./sarvam-tamil-therapy-fixed/final_model
 Saved!
 Also saved to Drive: /content/drive/MyDrive/sarva-tamil-therapy


##  CELL 9: Test Generation (FIXED - No more 'user' output!)

In [ ]:
print("="*70)
print("TESTING TAMIL GENERATION (FIXED!)")
print("="*70)

def generate_tamil_response(user_input, emotion=None):
    """
    FIXED generation function.
    This will NOT output 'user' anymore!
    """
    # Add emotion if provided
    if emotion:
        user_input = f"[உணர்வு: {emotion}] {user_input}"

    # Build prompt
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are an empathetic Tamil therapy assistant. Respond in Tamil with care and understanding.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_input}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,      # INCREASED
            min_new_tokens=40,       # FORCE minimum
            temperature=0.8,         # Slightly higher
            top_p=0.92,             # Adjusted
            top_k=50,
            do_sample=True,
            repetition_penalty=1.15, # Prevent loops
            length_penalty=1.0,      # Encourage longer
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ROBUST EXTRACTION (prevents 'user' bug)
    # Remove everything before assistant response
    if 'assistant' in response:
        parts = response.split('assistant')
        if len(parts) > 1:
            response = parts[-1].strip()

    # Remove any trailing tokens
    if '<|eot_id|>' in response:
        response = response.split('<|eot_id|>')[0].strip()

    # Remove system/user text if present
    if 'system' in response.lower():
        response = response.split('system', 1)[-1].strip()
    if 'user' in response.lower() and len(response) < 20:
        # If response is just 'user', generate again
        return "[Generation failed - retrying needed]"

    return response.strip()

# TEST CASES
print("\n Testing...\n")

tests = [
    ("எனக்கு மிகவும் வருத்தமாக உள்ளது", "வருத்தம்"),
    ("எனக்கு கவலையாக இருக்கிறது", "கவலை"),
    ("நான் தனிமையாக உணர்கிறேன்", "தனிமை"),
    ("எனக்கு கோபமாக இருக்கிறது", "கோபம்"),
]

for i, (text, emotion) in enumerate(tests, 1):
    print(f"Test {i}/{len(tests)}:")
    print(f"User: {text}")

    response = generate_tamil_response(text, emotion=emotion)

    print(f"Bot: {response}")
    print(f"Length: {len(response.split())} words")

    # Check quality
    has_tamil = any('\u0B80' <= c <= '\u0BFF' for c in response)
    is_complete = len(response) > 50

    print(f"Tamil: {'Yes' if has_tamil else 'No'}")
    print(f"Complete: {'Yes' if is_complete else 'No'}")
    print("-" * 70)
    print()

print("Testing done!")

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


TESTING TAMIL GENERATION (FIXED!)

 Testing...

Test 1/4:
User: எனக்கு மிகவும் வருத்தமாக உள்ளது


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: <|end_header_id|>

அ <| <| <|<| <| <||> <| <| << <| <|<< <| <|)| <| <||| <| <|ైర్ <| <|| <| <| | <| <| {" <| <|!< <| <|))-> <| <|ంటీ <| <|ற்கு <| <|norm <| <| `< <| <| celebrated <| <| States <| <|pathy <| <| ਹਾਂ <| <|ନ୍ତି <| <|ually <| <|.|| <| <|__, <| <|"\ <| <| ઓસ્ટ્રે <| <|।'' <| <| hygiene <| <|abetes <| <| //-------- <| <| () <| <| sever <| <| (- <| <|adelph <| <| Joh <| <| bene <| <| purchased <| <|_: <| <|cember <| <| England <| <|ੰਸ਼ <| <| "( <| <| '_ <| <| --------- <| <|sem <| <| ସମ <| <| @@ <| <||>' <| <|లికన్ <| <| || <| <|Link <| <|________, <| <|ফেন <| <|શ્ચર્ય <| <| < <| <| compon <| <| |= <| <|:|| <| <|{" <| <| ਤੁਰ <| <| Kent <| <| arthritis <| <| "$ <| <|gest <| <|ନିଆ <| <| Os <| <|bestos <| <|న్నిస్ <| <| organized <| <| (' <| <| Commons <| <| absorbed <| <|ଫେ <| <| walking <| <| { <| <||>" <| <|{ <| <| :: <| <|_{ <| <| alle <| <|(- <| <|member <| <| interpre <| <| discussing <| <|શેષ <| <| experienced <| <|^ <| <| flies <| <|ानुसार <| <| acute <| <| ഒക്ടോ <| <

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: <|end_header_id|>

அ <| <| <|<| <| <| << <| <|<< <| <||> <| <|| <| <|)| <| <| //-------- <| <||| <| <|_: <| <| ਹਾਂ <| <|))-> <| <| England <| <|ற்கு <| <| `< <| <| | <| <| celebrated <| <|.|| <| <|ావి <| <| (- <| <| "( <| <|ନ୍ତି <| <|ైర్ <| <|(- <| <|।'' <| <|{" <| <|abetes <| <|sem <| <|< <| <| "$ <| <|੍ਹਾ <| <|adelph <| <|શ્ચર્ય <| <| compon <| <|ంటీ <| <|ନିଆ <| <|!< <| <| absorbed <| <| (' <| <|:|| <| <| ઓસ્ટ્રે <| <| --------- <| <|pathy <| <| < <| <| States <| <| hygiene <| <| "- <| <|>, <| <| bene <| <|"\ <| <| sever <| <|lims <| <| @@ <| <|શેષ <| <| () <| <| observed <| <| {" <| <|లికన్ <| <| :: <| <| ਤੁਰ <| <| purchased <| <|strong <| <| || <| <| Joh <| <|ds <| <|ੰਸ਼ <| <| placing <| <| Democratic <| <| কেউ <| <| acute <| <|Link <| <| alle <| <|norm <| <| '_ <| <| कोणी <| <| Kent <| <|[/INST] <| <|bestos <| <| ਪਲਾ <| <| arthritis <| <|ilities <| <| walking <| <| celebrate <| <|member <| <|ফেন <| <| manuscript <| <| passes <| <|.| <| <|� <| <| iv <| <|.( <| <|gest <| <||>' 

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: <|end_header_id|>

த <| <| <||> <| <|<| <| <|<< <| <|)| <| <| << <| <||| <| <|ావి <| <|))-> <| <|ନ୍ତି <| <|adelph <| <|.|| <| <|_: <| <|ంటీ <| <| | <| <|!< <| <|sem <| <|pathy <| <|(- <| <| (- <| <| hygiene <| <|ైర్ <| <| ਤੁਰ <| <|ற்கு <| <| ਹਾਂ <| <|:|| <| <| sever <| <|__, <| <|| <| <| || <| <| compon <| <| "( <| <| celebrated <| <|లికన్ <| <|"\ <| <| observed <| <|________, <| <|।'' <| <| //-------- <| <| alle <| <| England <| <|< <| <| {" <| <| purchased <| <|^ <| <|strong <| <|cember <| <||>' <| <|શ્ચર્ય <| <| (' <| <| States <| <| हस्त <| <|abetes <| <| { <| <|న్నిస్ <| <| Commons <| <| "$ <| <|norm <| <| ઓસ્ટ્રે <| <| ഒക്ടോ <| <|_{ <| <| ସମ <| <|ੰਸ਼ <| <| bene <| <| placing <| <| Os <| <|member <| <| arthritis <| <| --------- <| <| '_ <| <| () <| <| कोणी <| <| `< <| <| passing <| <|gest <| <|.( <| <|શેષ <| <| Kent <| <| everyone <| <|ফেন <| <| |= <| <|_( <| <| flies <| <| iv <| <|cat <| <|੍ਹਾ <| <| :: <| <| interpre <| <| organized <| <| somew <| <| "- <| <|>> <| <| @@ <| <

##  CELL 10: Full Evaluation

In [ ]:
print("="*70)
print("FULL EVALUATION")
print("="*70)

import evaluate

# Generate on validation
print("\nGenerating predictions...")
predictions = []
references = []
emotions = []

for i, ex in enumerate(val_dataset_eval):
    user_input = ex['user_input']
    ref = ex['target_response']
    emotion = ex['emotion']

    pred = generate_tamil_response(user_input, emotion=emotion if emotion != 'unknown' else None)

    predictions.append(pred)
    references.append(ref)
    emotions.append(emotion)

    if (i + 1) % 10 == 0:
        print(f"   {i + 1}/{len(val_dataset_eval)}...")

print(f"\n {len(predictions)} predictions")

# ROUGE
rouge = evaluate.load('rouge')
rouge_results = rouge.compute(predictions=predictions, references=references)

print("\n ROUGE:")
print(f"   ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"   ROUGE-L: {rouge_results['rougeL']:.4f}")

# Quality
avg_len = np.mean([len(p.split()) for p in predictions])
has_tamil = [any('\u0B80' <= c <= '\u0BFF' for c in p) for p in predictions]
tamil_pct = (sum(has_tamil) / len(has_tamil)) * 100
complete = [len(p) > 50 for p in predictions]
complete_pct = (sum(complete) / len(complete)) * 100

print("\n Quality:")
print(f"   Avg length: {avg_len:.1f} words")
print(f"   Tamil: {tamil_pct:.1f}%")
print(f"   Complete: {complete_pct:.1f}%")

# Show examples
print("\n Examples:")
for i in range(min(5, len(predictions))):
    print(f"\n{i+1}. User: {val_dataset_eval[i]['user_input'][:50]}...")
    print(f"   Pred: {predictions[i][:100]}...")

print("\n" + "="*70)
print(" EVALUATION DONE!")
print("="*70)

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FULL EVALUATION

Generating predictions...


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   10/98...


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   20/98...


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   30/98...


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

KeyboardInterrupt: 